In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr

# Load data
issue_features = pd.read_csv('../data/issue_features.csv')
gallup = pd.read_csv('/Users/pranshutashukla/Documents/GML/data/gallup_ground_truth.csv')
rgcn_results = pd.read_csv('../data/rgcn_dissonance.csv')

# Clean gallup
gallup = gallup.rename(columns={gallup.columns[0]: 'issue'})
gallup['issue'] = gallup['issue'].str.strip()
gallup['importance_score'] = gallup['% Extremely important'] + 0.5 * gallup['% Very important']

print("Issue features shape:", issue_features.shape)
print("Issue features columns:", issue_features.columns.tolist())
print("\nGallup shape:", gallup.shape)
print("\nR-GCN results shape:", rgcn_results.shape)

Issue features shape: (22, 12)
Issue features columns: ['issue', 'Tweets', 'Users', 'avg_user_degree', 'avg_hashtag_degree', 'avg_pagerank', 'avg_betweenness', 'num_conversations', 'conv_depth', 'spread', 'avg_betweenness_scaled', 'avg_pagerank_scaled']

Gallup shape: (22, 6)

R-GCN results shape: (22, 5)


In [2]:
# Feature columns for MLP
feature_cols = ['Tweets', 'Users', 'avg_user_degree', 'avg_hashtag_degree', 
                'avg_pagerank', 'avg_betweenness', 'num_conversations', 'conv_depth', 'spread']

# Merge with gallup to align issues
df = issue_features.merge(gallup[['issue', 'importance_score']], on='issue', how='left')

# Normalize features
from sklearn.preprocessing import StandardScaler

# MLP predicts amplification score, not Gallup importance
rgcn_results_aligned = rgcn_results.merge(df[['issue'] + feature_cols], on='issue', how='left')

X = rgcn_results_aligned[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float)

# Target is amplification score
y_amp = torch.tensor(rgcn_results_aligned['amplification_score'].values, dtype=torch.float)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Target shape: {y_amp.shape}")
print(f"\nIssues in order:\n{df['issue'].tolist()}")

Feature matrix shape: (22, 9)
Target shape: torch.Size([22])

Issues in order:
['Democracy in the U.S.', 'Crime', 'Immigration', 'Types of Supreme Court justices candidates would pick', 'The federal budget deficit', 'Situation in Middle East between Israelis and Palestinians', 'Gun policy', 'Relations with China', 'Climate change', 'Transgender rights', 'The economy', 'Abortion', 'Terrorism and national security', 'Distribution of income and wealth in the U.S.', 'Taxes', 'Foreign affairs', 'Race relations', 'Energy policy', 'Education', 'Relations with Russia', 'Healthcare', 'Trade with other nations']


### Define & Train MLP

In [3]:
# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float)
# y_tensor = torch.tensor(y, dtype=torch.float)

# Define MLP
# class MLP(nn.Module):
#     def __init__(self, in_channels, hidden_channels):
#         super().__init__()
#         self.fc1 = nn.Linear(in_channels, hidden_channels)
#         self.fc2 = nn.Linear(hidden_channels, hidden_channels)
#         self.fc3 = nn.Linear(hidden_channels, 1)

#     def forward(self, x):
#         x = F.relu(self.fc1(x))
#         x = F.dropout(x, p=0.1, training=self.training)
#         x = F.relu(self.fc2(x))
#         x = self.fc3(x)
#         return x.squeeze()
class MLP(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, hidden_channels)
        self.fc3 = nn.Linear(hidden_channels, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x.squeeze()

model_mlp = MLP(in_channels=9, hidden_channels=32)
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

# Training
losses = []
for epoch in range(1, 501):
    model_mlp.train()
    optimizer.zero_grad()
    pred = model_mlp(X_tensor)
    loss = F.mse_loss(pred, y_amp)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

Epoch 050 | Loss: 1.5817
Epoch 100 | Loss: 0.2988
Epoch 150 | Loss: 0.1026
Epoch 200 | Loss: 0.0513
Epoch 250 | Loss: 0.0233
Epoch 300 | Loss: 0.0112
Epoch 350 | Loss: 0.0060
Epoch 400 | Loss: 0.0028
Epoch 450 | Loss: 0.0010
Epoch 500 | Loss: 0.0003


In [4]:
# Plot loss curve
import plotly.graph_objects as go

epochs_to_plot = list(range(1, 301, 5))
losses_to_plot = [losses[i-1] for i in epochs_to_plot]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=epochs_to_plot,
    y=losses_to_plot,
    mode='lines+markers',
    line=dict(color='#7F77DD', width=2),
    marker=dict(size=4),
    name='Training Loss'
))

fig.update_layout(
    title='MLP Training Loss',
    xaxis_title='Epoch',
    yaxis_title='Loss (log scale)',
    template='plotly_white',
    height=400
)
fig.show()

### MLP Amplification Score & Spearman Correlation

In [7]:
# Get MLP predictions
model_mlp.eval()
with torch.no_grad():
    mlp_preds = model_mlp(X_tensor).numpy()

# Build MLP results dataframe
mlp_df = pd.DataFrame({
    'issue': rgcn_results_aligned['issue'],
    'mlp_score': mlp_preds,
    'importance_score': rgcn_results_aligned['importance_score']
})

mlp_df = mlp_df.sort_values('mlp_score', ascending=False).reset_index(drop=True)
mlp_df['mlp_rank'] = range(1, len(mlp_df) + 1)

# Spearman correlation
gallup_ranks = mlp_df['importance_score'].rank(ascending=False).astype(int)
corr_mlp, pvalue_mlp = spearmanr(gallup_ranks, mlp_df['mlp_rank'])

print(f"MLP Spearman Correlation: {corr_mlp:.4f}")
print(f"R-GCN Spearman Correlation: 0.2791")
print(f"Difference in Correlation: {corr_mlp - 0.2791:.4f}")

print(f"\nMLP P-value: {pvalue_mlp:.4f}")
print(f"R-GCN P-value: 0.2084")
print(f"Difference in P-value: {pvalue_mlp - 0.2084:.4f}")

MLP Spearman Correlation: 0.2972
R-GCN Spearman Correlation: 0.2791
Difference in Correlation: 0.0181

MLP P-value: 0.1792
R-GCN P-value: 0.2084
Difference in P-value: -0.0292


In [8]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name='R-GCN',
    x=['Spearman Correlation'],
    y=[0.2791],
    marker_color='#378ADD',
    text=['0.2791'],
    textposition='outside'
))

fig.add_trace(go.Bar(
    name='MLP',
    x=['Spearman Correlation'],
    y=[0.2972],
    marker_color='#7F77DD',
    text=['0.2972'],
    textposition='outside'
))

fig.update_layout(
    title='Model Comparison: Spearman Correlation with Gallup Rankings',
    yaxis_title='Spearman Correlation',
    yaxis=dict(range=[0, 0.5]),
    template='plotly_white',
    barmode='group',
    height=400
)
fig.show()

In [14]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Spearman Correlation',
    x=['R-GCN', 'MLP'],
    y=[0.2791, 0.2972],
    marker_color=['#378ADD', '#7F77DD'],
    text=['0.2791', '0.2972'],
    textposition='outside'
))

fig.add_trace(go.Bar(
    name='P-value',
    x=['R-GCN', 'MLP'],
    y=[0.2084, 0.1792],
    marker_color=['#D85A30', '#EF9F27'],
    text=['0.2084', '0.1792'],
    textposition='outside'
))

fig.add_hline(
    y=0.05,
    line_dash='dash',
    line_color='red',
    annotation_text='p=0.05 significance threshold',
    annotation_position='top right'
)

fig.update_layout(
    title='Model Comparison: R-GCN vs MLP',
    xaxis_title='Model',
    yaxis_title='Score',
    yaxis=dict(range=[0, 0.5]),
    template='plotly_white',
    barmode='group',
    height=450
)
fig.show()